In [ ]:
import numpy as np
import pandas as pd

In [6]:
products = pd.read_csv("products_data.csv")
products.head(2)

,product_id,product_name,category,manufacturer
0,101,Ноутбук HP Pavilion 15,Ноутбуки,A
1,102,Смартфон Samsung Galaxy S21,Смартфоны,B


In [7]:
sales = pd.read_csv("sales_data.csv")
sales.head(2)

,order_id,product_id,customer_id,order_date,quantity,price_per_unit,total_price,payment_method,region
0,1,101,1001,2022-02-15,2,500,1000,Карта,Север
1,2,102,1002,2022-03-20,1,800,800,Наличные,Юг


In [31]:
df_o = pd.merge(
    sales, products, on="product_id", how="outer", suffixes=("_sal", "_prod")
)
df_prods = pd.merge(
    sales, products, on="product_id", how="right", suffixes=("_sal", "_prod")
)
df_sales = pd.merge(
    sales, products, on="product_id", how="left", suffixes=("_sal", "_prod")
)

In [32]:
df_o.head(2)
df_prods.head(2)
df_sales.head(2)

,order_id,product_id,customer_id,order_date,quantity,price_per_unit,total_price,payment_method,region,product_name,category,manufacturer
0,1,101,1001,2022-02-15,2,500,1000,Карта,Север,Ноутбук HP Pavilion 15,Ноутбуки,A
1,2,102,1002,2022-03-20,1,800,800,Наличные,Юг,Смартфон Samsung Galaxy S21,Смартфоны,B


In [12]:
df_1 = (
    df.groupby(["region", "category"])
    .agg(items_sold=("quantity", "sum"))
    .reset_index()
    .query('category == "Смартфоны"')
    .sort_values("items_sold", ascending=False)
)

df_1

,region,category,items_sold
3,Восток,Смартфоны,4
18,Юг,Смартфоны,3
8,Запад,Смартфоны,2
13,Север,Смартфоны,1


In [20]:
df_2 = (
    df.query('category == "Ноутбуки"')
    .groupby("manufacturer")["price_per_unit"]
    .mean()
    .agg(lambda x: x.max() - x.min())
)

df_2

np.float64(200.0)

In [30]:
df_3 = df.groupby("customer_id")["total_price"].sum().sort_values(ascending=False)

df_3

customer_id
1003    3550
1007    3500
1001    3150
1006    2850
1009    2650
1010    2050
1011    2050
1002    2000
1004    1900
1008    1750
1015    1350
1005    1200
1014    1200
1012     900
1016     800
1018     700
1013     550
1017     300
Name: total_price, dtype: int64

In [37]:
planshet_rev_share = (
    df_prods[df_prods["category"] == "Планшеты"]["total_price"].sum()
    / df_prods["total_price"].sum()
).round(2)

planshet_rev_share

np.float64(0.25)

Какова абсолютная разница в средней цене за единицу продуктов категории "Смартфоны" между двумя регионами с наибольшей общей выручкой по всем категориям товаров (учитывая только продукты, информация о которых есть в products_data)? Ответ округлите до целых и возьмите по модулю.

In [42]:
top_regions = df_prods.groupby("region")["total_price"].sum().nlargest(2).index

result = (
    df_prods[
        (df_prods["category"] == "Смартфоны") & (df_prods["region"].isin(top_regions))
    ]
    .groupby("region")["price_per_unit"]
    .mean()
    .pipe(lambda x: abs(x.iloc[0] - x.iloc[1]))
)

result

np.float64(275.0)

Сколько в среднем уникальных категорий продуктов было продано по всем регионам?

In [44]:
result = df_prods.groupby("region")["category"].apply("nunique").mean()

result

np.float64(5.0)

Какой производитель представлен в наибольшем количестве различных категорий продуктов?

In [47]:
products.groupby("manufacturer")["category"].apply("nunique").sort_values(
    ascending=False
)

manufacturer
E    4
C    3
B    3
D    3
A    2
Name: category, dtype: int64

В каком месяце было продано больше всего продуктов категории "Планшеты" в регионе "Север"? (ответ представьте в виде номера месяца)

In [52]:
res = (
    df_prods[(df_prods["category"] == "Планшеты") & (df_prods["region"] == "Север")]
    .groupby(pd.to_datetime(df_prods["order_date"]).dt.to_period("M"))["total_price"]
    .agg("sum")
)

res


order_date
2022-08    900
Freq: M, Name: total_price, dtype: int64

Какой производитель продал продукты на наибольшую сумму в регионе "Юг" за март 2023 года?

In [57]:
res = (
    df_prods[
        (df_prods["region"] == "Юг")
        & (pd.to_datetime(df_prods["order_date"]).dt.to_period("M") == "2023-03")
    ]
    .groupby("manufacturer")["total_price"]
    .sum()
    .nlargest(1)
)

res

manufacturer
B    700
Name: total_price, dtype: int64

Сколько покупателей совершили заказы на продукты хотя бы трех разных категорий?

In [70]:
res_10 = (
    df_prods.groupby("customer_id")["category"]
    .agg("nunique")
    .reset_index()
    .groupby("category")
    .size()
    # .query("category >= 3")
)

res_10

category
1    10
2     7
3     1
dtype: int64

Какова доля продаж продуктов производителя "C" от общей выручки за все время в регионе "Север"? Ответ округлите до 2 знака после запятой. В качестве разделителя разрядов используйте точку.

In [72]:
sales[sales["region"] == "Север"]["total_price"].sum()

np.int64(10200)

In [76]:
res_11 = (
    df_prods.query('manufacturer=="C" and region=="Север"')["total_price"].sum()
    / df_prods[df_prods["region"] == "Север"]["total_price"].sum()
)

res_11.round(2)

np.float64(0.27)

айдите покупателей, которые купили более чем 15% уникальных связок "Категория - Производитель". Если таких покупателей несколько - напишите их через запятую с пробелом в порядке возрастания результата.

In [98]:
total_pairs = (
    df_prods.groupby(["category", "manufacturer"])["manufacturer"].agg("nunique").sum()
)

unique_pairs_per_customer = df_prods.groupby(
    ["customer_id", "category", "manufacturer"]
)["manufacturer"].agg("nunique")

share_of_pairs_per_customer = (
    (
        unique_pairs_per_customer.groupby("customer_id")
        .sum()
        .apply(lambda x: x / total_pairs)
    )
    .sort_values(ascending=False)
    .reset_index()
    .query("manufacturer > 0.15")["customer_id"]
    .tolist()
)

share_of_pairs_per_customer

[1001, 1003]

Какое максимальное количество продуктов было продано в один заказ?

In [101]:
sales.groupby("order_id")["quantity"].sum().nlargest(1)

order_id
12    4
Name: quantity, dtype: int64

Какой производитель имеет самую высокую среднюю цену продукта среди всех категорий?

In [112]:
df_prods.groupby(["manufacturer", "category"])["price_per_unit"].mean().nlargest(1)

# .reset_index(level=1)

manufacturer  category
E             Наушники    1000.0
Name: price_per_unit, dtype: float64

Каково общее количество уникальных производителей, чьи продукты были куплены менее чем в 3 регионах?

In [114]:
df_prods.groupby("manufacturer")["region"].nunique()

manufacturer
A    4
B    4
C    4
D    4
E    4
Name: region, dtype: int64